# 03 · Train Model B — the surrogate panel

Deliberately *not* hardened. Trained on the non-adversarial split only, so it
behaves like a typical third-party detector — which is the whole point.

The humanizer optimises against this model plus the perplexity ratio and the
stylometric features. Optimising against a panel rather than a single
classifier is what makes the rewrite generalise: tuned against one model it
learns that model's quirks; tuned against a diverse panel it learns the
properties third-party detectors genuinely share (PRD 8.1).

Same architecture, same loss, different data. **Needs GPU**, 4–8 hours.


In [ ]:
# Setup. Run once per session — everything below depends on it.
# Version ranges, not open-ended ones. An unbounded "transformers>=4.44"
# silently upgrades to whatever released since the last session, over the
# build Kaggle already tested, and DeBERTa-v3 then stops loading with no
# code change at all. Pinning below 5 keeps a run reproducible.
!pip install -q "transformers>=4.44,<5" "tokenizers>=0.19,<0.22" \
    "datasets>=2.20,<4" sentencepiece onnx onnxruntime \
    "optimum[onnxruntime]>=1.20,<2" pyarrow \
    google-api-python-client google-auth google-auth-oauthlib

import sys, os
from pathlib import Path

# Must be set before torch is imported anywhere — PyTorch reads it when CUDA
# first initialises. Reduces the fragmentation that turns "enough memory" into
# an out-of-memory error hours into a run.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Change this if you forked the repo. Public repo => no token needed.
GIT_URL = "https://github.com/ByteCraft-9/ai-text-humanizer.git"

# Either attached as a Kaggle Dataset named `ai-detector-repo`, or cloned.
REPO = Path("/kaggle/input/ai-detector-repo") if Path("/kaggle/input/ai-detector-repo").exists() \
       else Path("/kaggle/working/ai-detector")
if not REPO.exists():
    !git clone --depth 1 $GIT_URL /kaggle/working/ai-detector

sys.path.insert(0, str(REPO / "training"))
sys.path.insert(0, str(REPO / "api"))

# parents=True so this also works off-Kaggle (Colab, a local box) after
# pointing WORK somewhere that exists.
WORK = Path("/kaggle/working"); WORK.mkdir(parents=True, exist_ok=True)
DATA = WORK / "data"; DATA.mkdir(parents=True, exist_ok=True)
MODELS = WORK / "models"; MODELS.mkdir(parents=True, exist_ok=True)

print("repo:", REPO)
print("work:", WORK)
assert (REPO / "training" / "lib").is_dir(), "repo not found — check GIT_URL"

# Preflight the backbone. Stage 2 loads it only after reading 230k rows, so a
# broken transformers install otherwise surfaces minutes in. Five seconds here.
import transformers
from transformers.utils import is_torch_available

print("transformers:", transformers.__version__)
if not is_torch_available():
    raise RuntimeError(
        "transformers cannot see PyTorch, so none of its torch model classes "
        "are registered. Usually means pip changed torch or transformers "
        "under a running kernel: restart the session (Run -> Restart Session) "
        "and run this cell first."
    )
try:
    from transformers import DebertaV2Model  # noqa: F401
except Exception as exc:
    raise RuntimeError(
        f"transformers {transformers.__version__} does not expose "
        f"DebertaV2Model ({exc}). If pip upgraded transformers during this "
        f"session, the running kernel still holds the old one: restart the "
        f"session and run this cell before anything else."
    ) from None
print("DebertaV2Model: available")


In [ ]:
# ---------------------------------------------------------------------------
# Run configuration. Read this cell before starting anything else.
# ---------------------------------------------------------------------------
#
# Kaggle gives 30 GPU-hours a week. A full run is 8-16 of them, so you cannot
# afford to discover a bug at hour six. Leave SMOKE_TEST = True for the first
# pass: it runs the entire pipeline end to end in well under an hour on a
# tiny sample. If stage 4 completes, the chain works. Then set it False and
# run for real.

SMOKE_TEST = True

if SMOKE_TEST:
    SAMPLE_ROWS = 5_000     # rows per dataset
    EPOCHS = 1
else:
    SAMPLE_ROWS = 400_000   # PRD 12.2
    EPOCHS = 3

print(f"{'SMOKE TEST' if SMOKE_TEST else 'FULL RUN'}: "
      f"{SAMPLE_ROWS:,} rows/dataset, {EPOCHS} epoch(s)")
if not SMOKE_TEST:
    print("Expect ~1-2 h for stage 1, then 4-8 h per model. Use "
          "Save Version -> Save & Run All so a browser disconnect cannot kill it.")


In [ ]:
# ---------------------------------------------------------------------------
# Google Drive - the one place work survives the session ending.
# ---------------------------------------------------------------------------
#
# Kaggle wipes /kaggle/working when a session ends or times out. Everything
# expensive - the dataset, the newest checkpoint, the final model - is written
# once locally and uploaded here. There is no second copy and no second
# cadence; the file you see on disk is the file that gets uploaded.
#
# Use OAuth, NOT a service account. A service account has no Drive storage of
# its own and would own anything it creates, so uploads fail with
# "Service Accounts do not have storage quota" however the folder is shared.
# Google's suggested workarounds need Google Workspace. Authorising as
# yourself makes the files yours, counted against your own quota.
#
# One-time setup, on a machine with a browser (not here):
#   1. console.cloud.google.com -> new project -> enable the Drive API
#   2. APIs & Services -> OAuth consent screen -> External.
#      Set publishing status to "In production". While it says "Testing",
#      Google expires the refresh token after 7 days and saving would stop
#      working mid-run. No verification review is needed, because the only
#      scope requested is drive.file, which is not a sensitive scope.
#   3. Credentials -> Create OAuth client ID -> Desktop app -> download JSON
#   4. Run, from a clone of this repo:
#        pip install google-auth-oauthlib google-api-python-client
#        python scripts/drive_auth.py --client-secret client_secret.json
#      It opens a browser, then writes drive_token.json and creates the Drive
#      folder for you.
#   5. Upload drive_token.json to Kaggle as a PRIVATE dataset, and point at it
#      below. Keep it private: it can write to that folder.

DRIVE_KEY_PATH = ""   # e.g. "/kaggle/input/drive-token/drive_token.json"

import os

if DRIVE_KEY_PATH:
    os.environ["DRIVE_SERVICE_ACCOUNT_JSON"] = DRIVE_KEY_PATH
    # The folder id is recorded inside drive_token.json, so there is nothing
    # else to fill in. Override here only for a different folder.
    # os.environ["DRIVE_FOLDER_ID"] = "..."

from lib.store import build_store
STORE = build_store()

if STORE.__class__.__name__ == "NullStore":
    print("")
    print("Nothing will survive this session ending. Fine for a smoke test;")
    print("set DRIVE_KEY_PATH above before starting a real run.")
else:
    # Prove the credentials work now, rather than discovering they do not
    # eight hours in when the first checkpoint tries to upload.
    from pathlib import Path
    probe = Path("/kaggle/working/.drive_probe")
    probe.write_text("ok")
    if STORE.push(probe, "_probe.txt") and STORE.pull("_probe.txt", probe):
        print("Drive write + read verified - checkpoints will survive.")
    else:
        print("Drive is NOT working. The message above says why.")
    probe.unlink(missing_ok=True)


In [ ]:
import pandas as pd
from lib.data import FEATURE_NAMES
from lib.train import TrainConfig, train

frame_b = pd.read_parquet(DATA / "train_b.parquet")

# Hold out by *domain*, not at random. A random split lets the model memorise
# a generator's quirks and score well on rows from the same generator, which
# is precisely the overfitting RAID exposed (E3: fine-tuned RoBERTa-Large
# averaged 56.7%).
holdout_b = sorted(frame_b["domain"].unique())[-2:]
validation_b = frame_b[frame_b["domain"].isin(holdout_b)]
training_b = frame_b[~frame_b["domain"].isin(holdout_b)]
print(f"train {len(training_b):,} · validate {len(validation_b):,} "
      f"on {holdout_b}")

config_b = TrainConfig(
    backbone="microsoft/deberta-v3-base",
    max_length=768,
    batch_size=16,
    accumulation_steps=2,
    learning_rate=2e-5,
    epochs=EPOCHS,
    fp16=True,
)

# Checkpoints land in WORK/model_b every 500 steps, and every
# remote_sync_minutes the newest one is pushed to STORE. On startup this looks
# for a local checkpoint, then a remote one, and fast-forwards to the step it
# reached — so if the session dies, run the notebook from the top again and
# training picks up where it stopped.
model_b, report_b = train(
    training_b, validation_b, list(FEATURE_NAMES),
    output_dir=WORK / "model_b",
    config=config_b,
    store=STORE,
)


In [ ]:
# Sanity check on what B is *for*. It should be clearly weaker than A on
# humanized text — that gap is the product's honesty margin, and the two
# numbers the UI shows are exactly this difference made visible.
final_b = report_b["final"]
print(f"Model B  AUROC {final_b['auroc']:.4f}  ·  TPR@1%FPR {final_b['tpr_at_1_fpr']:.4f}")
try:
    print(f"Model A  AUROC {report_a['final']['auroc']:.4f}  ·  "
          f"TPR@1%FPR {report_a['final']['tpr_at_1_fpr']:.4f}")
except NameError:
    print("(Model A not in this session — compare against stage 2's output.)")
print("\nIf B matches A on adversarial text it is not a surrogate for anything.")
print("Stage 5 measures that gap directly.")
